In [1]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]

selected_campaigns = [1]
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)




Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [2]:
matrix = df.loc[1, "measurements_matrix"]


,pci,beam_index,nr_arfcn,operator_id,sinr,rsrq
27,57,3,432050,10,3.310000,-11.910000
39,-109,0,643296,10,-7.391817,-18.713237
40,-109,1,643295,10,-6.739257,-18.129174
41,-109,1,643296,10,0.198050,-13.285338
42,-109,2,643295,10,-4.980469,-16.554104
43,-109,2,643296,10,0.720351,-13.092068
44,-109,3,643295,10,-10.592826,-21.298510
45,-109,3,643296,10,-5.314371,-16.733598
46,-109,4,643296,10,-10.106859,-20.800786
47,-109,6,643296,10,-12.652067,-23.162181


In [19]:
import pandas as pd
from scripts.utils import extract_unique_npcis


def matrix_filter(
        mat: pd.DataFrame,
        rf_param: RF_PARAM_5G,
        include_n_best_pcis: int = None,
        include_n_best_beams: int = None,
):
    """

    :param mat: Measurements matrix
    :param rf_param: What RF parameter to sort points by
    :param include_n_best_pcis: How many PCIs to include, set to None to include all PCI points
    :param include_n_best_beams: How many Beams to include, set to None to include all Beams
    :return:
    """

    mat = mat.dropna(subset=[rf_param.value])

    if mat.empty: return mat

    if not (include_n_best_pcis or include_n_best_beams):
        return mat
    # Get the maximum value for each PCI and beam_index combination
    idx = mat.groupby(['pci', 'beam_index'])[rf_param.value].idxmax()
    beams = mat.loc[idx].sort_values(by=[rf_param.value], ascending=False)

    # Filter for the n best PCIs if specified
    if include_n_best_pcis:
        # Get the n best unique PCIs based on their maximum RF parameter value
        best_pcis = beams.groupby('pci')[rf_param.value].max().nlargest(include_n_best_pcis).index.tolist()
        beams = beams[beams['pci'].isin(best_pcis)]

    # Filter for the b best beams for each PCI if specified
    if include_n_best_beams:
        # For each PCI, get the b best beams
        beams = beams.groupby('pci').apply(
            lambda x: x.nlargest(include_n_best_beams, rf_param.value)
        ).reset_index(drop=True)

    return beams


unique_before = extract_unique_npcis(df['measurements_matrix'])

# filter the df matricies

df.loc[:, "measurements_matrix"] = df.loc[:, "measurements_matrix"].apply(
    lambda x: matrix_filter(
        x,
        rf_param=RF_PARAM_5G.RSRQ,
        include_n_best_pcis=None,
        include_n_best_beams=1,
    )
)

unique_after = extract_unique_npcis(df['measurements_matrix'])

print(f'Before {len(unique_before)}, After {len(unique_after)}')

unique_before

/var/folders/sz/wsd2tvj51v1c7jpt5gscdckc0000gn/T/ipykernel_27627/3268412740.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  beams = beams.groupby('pci').apply(
/var/folders/sz/wsd2tvj51v1c7jpt5gscdckc0000gn/T/ipykernel_27627/3268412740.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  beams = beams.groupby('pci').apply(
/var/folders/sz/wsd2tvj51v1c7jpt5gscdckc0000gn/T/ipykernel_27627/3268412740.py:3

Before 154, After 111


/var/folders/sz/wsd2tvj51v1c7jpt5gscdckc0000gn/T/ipykernel_27627/3268412740.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  beams = beams.groupby('pci').apply(
/var/folders/sz/wsd2tvj51v1c7jpt5gscdckc0000gn/T/ipykernel_27627/3268412740.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  beams = beams.groupby('pci').apply(
/var/folders/sz/wsd2tvj51v1c7jpt5gscdckc0000gn/T/ipykernel_27627/3268412740.py:3

[(57, 3, 432050, 10),
 (-109, 0, 643296, 10),
 (-109, 1, 643295, 10),
 (-109, 1, 643296, 10),
 (-109, 2, 643295, 10),
 (-109, 2, 643296, 10),
 (-109, 3, 643295, 10),
 (-109, 3, 643296, 10),
 (-109, 4, 643296, 10),
 (-109, 6, 643296, 10),
 (-108, 3, 643296, 10),
 (-108, 4, 643296, 10),
 (-108, 5, 643295, 10),
 (-108, 5, 643296, 10),
 (-108, 6, 643296, 10),
 (10, 0, 643296, 10),
 (10, 1, 643296, 10),
 (10, 5, 643296, 10),
 (10, 6, 643295, 10),
 (10, 6, 643296, 10),
 (10, 7, 643295, 10),
 (10, 7, 643296, 10),
 (-111, 3, 432050, 10),
 (-65, 6, 643296, 10),
 (75, 0, 643295, 10),
 (75, 0, 643296, 10),
 (75, 1, 643296, 10),
 (75, 2, 643295, 10),
 (75, 2, 643296, 10),
 (75, 3, 643295, 10),
 (75, 3, 643296, 10),
 (75, 4, 643295, 10),
 (75, 4, 643296, 10),
 (75, 5, 643296, 10),
 (75, 6, 643296, 10),
 (75, 7, 643296, 10),
 (76, 5, 643296, 10),
 (76, 6, 643296, 10),
 (76, 7, 643296, 10),
 (121, 1, 643296, 10),
 (-62, 4, 643296, 10),
 (-58, 2, 643296, 10),
 (-58, 3, 643296, 10),
 (-10, 3, 432050, 1

In [17]:
matrix[matrix['pci'] == 75]

,pci,beam_index,nr_arfcn,operator_id,sinr,rsrq
99,75,0,643295,10,-7.470026,-18.563773
100,75,0,643296,10,-2.085086,-14.583016
101,75,1,643296,10,-8.405034,-19.344171
102,75,2,643295,10,-11.226502,-21.911969
103,75,2,643296,10,-4.480163,-16.145415
104,75,3,643295,10,-4.970133,-16.575678
105,75,3,643296,10,0.140467,-13.305489
106,75,4,643295,10,-4.165435,-15.996716
107,75,4,643296,10,1.169994,-12.906174
108,75,5,643296,10,-9.786710,-20.500314


In [5]:
df['size'].sum()

np.int64(5280882)

In [6]:
df['campaign_id'].unique().shape

(79,)

In [13]:
import folium
import pandas as pd


def geo_plot_points(df: pd.DataFrame):
    """
    Plots given locations to a map (OpenStreetMap) that is viewable in broswer.
    Generates a file called 'map.html' in the current working directory.
    :param df:
    """
    # Create a map centered around the mean location
    m = folium.Map(location=[df["lat"].mean(), df["lng"].mean()], zoom_start=12)

    # Add CircleMarkers to the map
    for _, row in df.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lng"]],
            radius=1,  # Size of the marker
            color="blue",  # Border color of the marker
            fill=True,
            fill_color="blue",  # Fill color of the marker
            fill_opacity=0.6,
        ).add_to(m)

    # Save the map as an HTML file and open it in the browser
    m.save("map.html")


geo_plot_points(df.sample(3000))
